In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 16)
spark.conf.set("spark.sql.files.maxPartitionBytes", 134217728)
spark.conf.set("spark.sql.files.openCostInBytes", 134217728)

In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.utils import AnalysisException
import logging

logging.basicConfig(level=logging.INFO)

logger = logging.getLogger("BronzePipeline")

In [0]:
CATALOG = "main_workspace"
BRONZE_SCHEMA = "bronze"

spark.sql(f"USE CATALOG {CATALOG}")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
BASE_PATH = "s3://e-commerce-analysis-pipeline/processed"

DATASETS = {
    "customers": f"{BASE_PATH}/customer_parquet/",
    "order_items": f"{BASE_PATH}/order_items_parquet/",
    "payments": f"{BASE_PATH}/order_payment_dataset_parquet/",
    "orders": f"{BASE_PATH}/orders_datasets_parquet/",
    "product_category_translation": f"{BASE_PATH}/product_category_name_translation_parquet/",
    "products": f"{BASE_PATH}/product_dataset_parquet/",
    "sellers": f"{BASE_PATH}/sellers_datasets/"
}

In [0]:
def load_to_bronze(table_name, path):

    logger.info(f"Starting ingestion for {table_name}")

    try:

        df = spark.read.format("parquet").load(path)

        # Check if dataframe is empty (Spark Connect compatible)
        if df.limit(1).count() == 0:
            logger.warning(f"{table_name} dataset is empty")
            return

        df = df.withColumn("ingestion_timestamp", current_timestamp())

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
        )

        logger.info(f"{table_name} successfully loaded")

    except AnalysisException as ae:

        logger.error(f"Schema error while loading {table_name}")
        logger.error(str(ae))

    except Exception as e:

        logger.error(f"Unexpected error while loading {table_name}")
        logger.error(str(e))

In [0]:
for table, path in DATASETS.items():
    load_to_bronze(table, path)

In [0]:
spark.sql("SHOW TABLES IN main_workspace.bronze").show()

In [0]:
spark.sql("""
SELECT *
FROM main_workspace.bronze.orders
LIMIT 10
""").display()

In [0]:
spark.sql("""
DESCRIBE DETAIL main_workspace.bronze.orders
""").display()

In [0]:
spark.sql("""
DESCRIBE HISTORY main_workspace.bronze.orders
""").display()

# **--Silver Layer--**